# CLASSIFICATORE DI FRUTTI ESOTICI

**TropicTaste Inc.** è un'azienda leader nella distribuzione di frutti esotici che punta a ottimizzare
l'efficienza e la precisione nel processo di classificazione dei prodotti.

Questo progetto mira a sviluppare un modello di Machine Learning in grado di **automatizzare la classificazione
dei frutti** basandosi su caratteristiche numeriche (peso, diametro, lunghezza, durezza della buccia, dolcezza),
riducendo gli errori umani e migliorando la gestione dell'inventario.

## OBIETTIVO DEL PROGETTO

Sviluppare un modello di Machine Learning per la **classificazione automatica dei frutti esotici**
basata su caratteristiche numeriche, con i seguenti obiettivi specifici:

- Ridurre gli errori umani nel processo di classificazione
- Ottimizzare la gestione dell'inventario
- Migliorare l'efficienza operativa e la qualità del prodotto

**Algoritmo scelto: K-Nearest Neighbors (KNN)**

Il KNN è stato scelto perché:
1. È intuitivo: classifica un campione in base ai suoi vicini più simili, esattamente come farebbe un operatore umano
2. Non fa assunzioni sulla distribuzione dei dati (algoritmo non parametrico)
3. Funziona bene con dataset di dimensioni moderate e feature numeriche ben definite
4. Non richiede una vera fase di addestramento (lazy learner), rendendo il modello facilmente aggiornabile

## SEZIONE 1: IMPORTAZIONI E CONFIGURAZIONE

In [ ]:
# ============================================================
# LIBRERIE UTILIZZATE E MOTIVAZIONE DELLE SCELTE
# ============================================================

import pandas as pd          # Gestione e manipolazione del dataset tabellare
import numpy as np           # Operazioni numeriche (array, statistiche, z-score)
import matplotlib.pyplot as plt  # Visualizzazione grafici di base
import seaborn as sns        # Visualizzazione statistica avanzata (heatmap, boxplot, scatterplot)

from google.colab import files  # Necessario per il caricamento del file CSV in Google Colab

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, LabelEncoder
# StandardScaler: normalizza le feature (media=0, std=1).
# FONDAMENTALE per KNN perché usa distanze euclidee: senza scaling,
# feature con valori grandi (es. Peso in grammi) dominerebbero quelle piccole (es. Dolcezza 1-10)
# LabelEncoder: converte le etichette categoriche del target in numeri interi

# --- Suddivisione e validazione ---
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
# train_test_split: divide il dataset in training set (80%) e test set (20%)
# GridSearchCV: ricerca sistematica del miglior iperparametro K tramite cross-validation
# cross_val_score: calcola lo score medio su più fold di cross-validation

# --- Modelli ---
from sklearn.neighbors import KNeighborsClassifier       # Modello principale del progetto
from sklearn.tree import DecisionTreeClassifier          # Usato nel confronto tra algoritmi
from sklearn.ensemble import RandomForestClassifier      # Usato nel confronto tra algoritmi
from sklearn.svm import SVC                              # Usato nel confronto tra algoritmi
from sklearn.naive_bayes import GaussianNB               # Usato nel confronto tra algoritmi

# --- Metriche di valutazione ---
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# accuracy_score: percentuale di predizioni corrette
# classification_report: precision, recall, F1-score per ogni classe
# confusion_matrix: tabella degli errori di classificazione per classe

from scipy.stats import zscore  # Calcolo dello Z-score per l'analisi degli outlier

import warnings
warnings.filterwarnings('ignore')  # Sopprime warning non critici per una lettura più pulita

## SEZIONE 2: CARICAMENTO E PRIMA ESPLORAZIONE DEL DATASET

Prima di costruire qualsiasi modello, è essenziale esplorare i dati per:
1. Comprendere la struttura del dataset (righe, colonne, tipi di dato)
2. Verificare la qualità dei dati (valori nulli, duplicati, outlier)
3. Capire le distribuzioni delle variabili

Queste informazioni guidano tutte le scelte successive di preprocessing e modellazione.

In [ ]:
# Caricamento del file CSV tramite l'interfaccia di Google Colab.
# Questo metodo permette di caricare il file direttamente dalla macchina locale
# senza dover montare Google Drive, rendendo il notebook facilmente condivisibile.
upload_file = files.upload()

In [ ]:
# Estraggo automaticamente il nome del file caricato (evito di hardcodare il nome)
file_name = next(iter(upload_file))

# Caricamento del dataset in un DataFrame pandas
df = pd.read_csv(file_name)

# Prima visualizzazione delle righe iniziali per verificare che il caricamento sia corretto
df.head()

In [ ]:
# Verifico le dimensioni del dataset: (n_righe, n_colonne)
# Utile per capire se il dataset è abbastanza grande per addestrare un modello
df.shape

In [ ]:
# Panoramica dei tipi di dato e dei valori non-null per ogni colonna
# Permette di identificare subito se ci sono tipi di dato errati o valori mancanti
df.info()

In [ ]:
# Verifico i valori unici nella colonna target 'Frutto'
# Fondamentale per sapere quante classi ha il nostro problema di classificazione
df["Frutto"].unique()

In [ ]:
# Controllo dei valori null per ogni colonna.
# I valori mancanti richiederebbero imputation o eliminazione delle righe;
# in questo caso verifichiamo che il dataset sia completo.
df.isnull().sum()

In [ ]:
# Controllo dei duplicati.
# Righe duplicate potrebbero creare data leakage (stesso campione in training e test)
# e falsare le metriche di valutazione.
df.duplicated().sum()

## SEZIONE 3: ANALISI STATISTICA E RILEVAMENTO DEGLI OUTLIER

In [ ]:
# Statistiche descrittive delle variabili numeriche.
# Analizzo: media, deviazione standard, min/max e quartili.
# Questo mi permette di capire la dispersione dei dati e
# individuare potenziali outlier (valori molto distanti dalla media).
df.describe()

In [ ]:
# BOXPLOT: visualizzazione grafica della distribuzione di tutte le feature numeriche.
# Il boxplot mostra: mediana (linea centrale), IQR (box), valori entro 1.5*IQR (baffi)
# e outlier (punti oltre i baffi).
# È un primo strumento visivo per identificare feature con alta variabilità e outlier estremi.
plt.figure(figsize=(8, 6))
sns.boxplot(data=df.drop("Frutto", axis=1))  # Escludiamo la colonna target categorica
plt.xticks(rotation=45)
plt.title("Distribuzione delle feature numeriche - Boxplot")
plt.tight_layout()
plt.show()

**ANALISI DEL BOXPLOT**

* **Dispersione dei valori (Deviazione Standard):**
  * `Diametro medio (mm)`: std più alto del dataset, suggerisce forte dispersione e presenza di outlier
  * `Peso (g)`: alta std ma senza outlier evidenti, indica variazione ampia ma coerente tra le classi
  * `Durezza buccia (1-10)`: qualche outlier superiore visibile

* **Distribuzione (Mediana vs Media):**
  * Per `Diametro medio`: mediana significativamente più bassa della media → distribuzione asimmetrica con valori estremi
  * Per le altre feature: mediana e media sono vicine → distribuzione quasi simmetrica

Procediamo con un'analisi quantitativa degli outlier tramite **Z-score**.

In [ ]:
# ============================================================
# RILEVAMENTO OUTLIER CON Z-SCORE
# ============================================================
# Lo Z-score misura quante deviazioni standard un valore dista dalla media della sua feature.
# Formula: z = (x - media) / std
# Soglia standard: |z| > 3 identifica un outlier (solo il 0.27% dei dati in distribuzione normale)
# Scelta della soglia 3: bilanciamento tra sensibilità (trovare anomalie reali) e
# specificità (non falsi positivi su variazioni naturali tra frutti diversi)

def calcolo_outlier(col_name):
    z_score = zscore(df[col_name])
    outliers = df[abs(z_score) > 3]
    print(f"Outlier nella colonna '{col_name}' (|Z-score| > 3): {len(outliers)}")
    return outliers

In [ ]:
# Funzione per disegnare uno scatter plot con evidenziazione degli outlier in rosso.
# Utilizziamo scatter plot (invece del boxplot) per vedere la distribuzione
# dei singoli campioni e la posizione esatta degli outlier.
COLORI_FEATURE = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

def design_single_scatterplot(ax, col_name, color):
    outliers = calcolo_outlier(col_name)
    sns.scatterplot(y=df[col_name], x=range(len(df)), ax=ax, color=color, alpha=0.6, label="Dati")
    # Gli outlier vengono evidenziati in rosso con bordo nero per renderli ben visibili
    sns.scatterplot(y=outliers[col_name], x=outliers.index, ax=ax,
                    color="red", s=100, edgecolor="black", label="Outlier")
    ax.set_title(f"Scatter Plot - {col_name}")
    ax.set_xlabel("Campioni")
    ax.set_ylabel(col_name)
    ax.legend()

In [ ]:
# Creo la griglia di scatter plot in modo dinamico:
# il numero di righe si adatta automaticamente al numero di feature,
# garantendo che il codice funzioni anche se il dataset cambia
feature_cols = df.drop("Frutto", axis=1).columns
num_col = len(feature_cols)
num_righe = (num_col // 2) + (num_col % 2)  # +1 riga se numero di feature è dispari

fig, axes = plt.subplots(nrows=num_righe, ncols=2, figsize=(12, num_righe * 4))

for i, col_name in enumerate(feature_cols):
    row, col = i // 2, i % 2
    design_single_scatterplot(axes[row, col], col_name, COLORI_FEATURE[i % len(COLORI_FEATURE)])

# Rimuovo gli assi vuoti nel caso in cui il numero di feature sia dispari
for j in range(i + 1, num_righe * 2):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()

**ANALISI DEGLI SCATTER PLOT**

* **Peso (g)**: distribuzione ampia con cluster ben separati che corrispondono alle categorie di frutto. Nessun outlier: la variazione è ampia ma coerente.
* **Diametro medio (mm)**: 4 outlier identificati (punti rossi). Alcuni frutti hanno dimensioni anomale rispetto alla media. La distribuzione è asimmetrica verso l'alto.
* **Durezza buccia (1-10)**: 2 outlier identificati, probabilmente frutti con buccia insolitamente dura.
* **Lunghezza media (mm)**, **Dolcezza (1-10)**: nessun outlier rilevato; distribuzione uniforme con cluster distinti.

**Decisione**: gli outlier rilevati sono pochi (6 su 500 campioni) e probabilmente rappresentano variazioni naturali tra esemplari della stessa specie. Non li eliminiamo per non ridurre ulteriormente il dataset.

## SEZIONE 4: ENCODING DELLE VARIABILI CATEGORICHE E MATRICE DI CORRELAZIONE

In [ ]:
# ============================================================
# LABEL ENCODING DELLA VARIABILE TARGET
# ============================================================
# Il KNN lavora con distanze numeriche, quindi la variabile target 'Frutto'
# (che è categorica) deve essere convertita in numeri interi.
#
# Scelta: LabelEncoder (invece di OneHotEncoder)
# Motivazione: il target è usato solo per la classificazione, non come feature.
# Il LabelEncoder assegna semplicemente un ID numerico a ogni classe;
# non c'è il rischio che il modello interpreti l'ordine numerico come ordinamento
# semantico (problema che invece sorge se si usa LabelEncoder su feature di input).

encoder = LabelEncoder()
df["Frutto"] = encoder.fit_transform(df["Frutto"])
df.head()

In [ ]:
# Visualizzo il mapping classe → numero per riferimento futuro
# (utile per interpretare la matrice di confusione)
mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Mapping frutti → codice numerico:")
for frutto, codice in mapping.items():
    print(f"  {frutto}: {codice}")

In [ ]:
# ============================================================
# MATRICE DI CORRELAZIONE
# ============================================================
# La matrice di correlazione (Pearson) mostra la relazione lineare tra coppie di variabili.
# Valori vicini a +1: correlazione positiva forte
# Valori vicini a -1: correlazione negativa forte
# Valori vicini a 0: nessuna relazione lineare
#
# Uso: identificare feature ridondanti (altamente correlate tra loro)
# e capire quali feature sono più correlate con il target

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), cbar=True, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice di Correlazione tra le variabili")
plt.tight_layout()
plt.show()

**ANALISI DELLA MATRICE DI CORRELAZIONE**

**Correlazioni con il target (Frutto):**
- `Diametro medio (-0.51)`: correlazione più forte con il target. Frutti con diametro maggiore tendono ad appartenere a classi con codice numerico più basso.
- Le altre feature hanno correlazioni più deboli con il target ma contribuiscono comunque alla distinzione tra le classi.

**Correlazioni tra feature (rischio ridondanza):**
- `Peso ↔ Lunghezza media (+0.58)`: correlazione positiva moderata-alta. I frutti più pesanti tendono ad essere più lunghi.
- `Diametro medio ↔ Dolcezza (-0.57)`: correlazione negativa. Frutti con diametro maggiore tendono ad essere meno dolci.
- `Lunghezza media ↔ Durezza buccia (-0.50)`: frutti più lunghi tendono ad avere buccia meno dura.

**Conclusione**: la correlazione Peso↔Lunghezza potrebbe suggerire ridondanza, ma testeremo entrambe le combinazioni per verificare empiricamente quale configurazione performa meglio.

## SEZIONE 5: PREPARAZIONE DEI DATI E DEFINIZIONE DEL MODELLO BASE

In [ ]:
# ============================================================
# CONFIGURAZIONE DELLA RICERCA DEGLI IPERPARAMETRI
# ============================================================

# Range di K da esplorare: da 1 a 100
# - K=1: overfitting (il modello memorizza i dati, sensibile al rumore)
# - K molto grande: underfitting (il modello è troppo generico)
# - Il valore ottimale è da trovare empiricamente con GridSearchCV
param_grid = {'n_neighbors': range(1, 101)}


def split_and_scale_data(X, y, random_state=42):
    """
    Suddivide il dataset in train/test, normalizza le feature e
    trova il miglior K per KNN tramite GridSearchCV con 5-fold cross-validation.

    Scelte metodologiche:
    - test_size=0.2: 80% training, 20% test. Proporzione standard che
      garantisce abbastanza dati per addestrare (400 campioni) e
      un test set significativo (100 campioni).
    - random_state=42: seed fisso per riproducibilità dei risultati.
    - StandardScaler: FONDAMENTALE per KNN. Normalizza ogni feature a
      media=0 e std=1, evitando che feature con valori numericamente
      grandi (es. Peso in grammi) dominino la distanza euclidea.
      ATTENZIONE: il fit dello scaler avviene SOLO sul training set per
      evitare data leakage (non vogliamo che le statistiche del test set
      influenzino la normalizzazione).
    - cv=5: 5-fold cross-validation. Bilanciamento tra affidabilità della
      stima (più fold = stime più stabili) e costo computazionale.
      Con 400 campioni di training, 5 fold da ~80 campioni ciascuno
      è una scelta robusta.
    - scoring='accuracy': metrica di ottimizzazione. Appropriata perché
      il dataset è bilanciato (100 campioni per classe).
    """
    # Suddivisione train/test con stratificazione implicita grazie al dataset bilanciato
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    # Normalizzazione: fit SOLO su X_train, transform su entrambi
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Creazione del modello KNN base e ricerca del K ottimale
    knn = KNeighborsClassifier()
    grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    print(f"Miglior K trovato:      {grid_search.best_params_['n_neighbors']}")
    print(f"Accuracy (5-fold CV):   {grid_search.best_score_:.4f}")

    return grid_search, X_train, X_test, y_train, y_test

## SEZIONE 6: TEST 1-4 — SELEZIONE DELLE FEATURE

Testiamo diverse combinazioni di feature per verificare empiricamente quale configurazione
offre le migliori performance. Questo processo è detto **Feature Selection**.

**Motivazione**: dalla matrice di correlazione sappiamo che alcune feature sono correlate
tra loro (es. Peso e Lunghezza con r=0.58). È possibile che rimuovere feature ridondanti
migliori le performance o le semplifichi senza perdita di accuratezza.

| Test | Feature Rimosse | Ipotesi |
|------|----------------|--------|
| 1 | Nessuna (tutte le feature) | Baseline |
| 2 | Peso (g) | Potenzialmente ridondante con Lunghezza |
| 3 | Lunghezza media (mm) | Potenzialmente ridondante con Peso |
| 4 | Dolcezza (1-10) | Meno correlata con il target |

### TEST 1: Tutte le feature (Baseline)
Utilizziamo tutte e 5 le feature disponibili come punto di riferimento.

In [ ]:
X = df.drop("Frutto", axis=1)   # Tutte le feature numeriche
y = df["Frutto"]                  # Target

grid_search, X_train, X_test, y_train, y_test = split_and_scale_data(X, y)

### TEST 2: Rimozione di 'Peso (g)'
Il Peso ha correlazione +0.58 con la Lunghezza: potrebbe essere ridondante.

In [ ]:
X2 = df.drop(["Frutto", "Peso (g)"], axis=1)
y2 = df["Frutto"]

grid_search2, X_train2, X_test2, y_train2, y_test2 = split_and_scale_data(X2, y2)

### TEST 3: Rimozione di 'Lunghezza media (mm)'
La Lunghezza ha correlazione -0.50 con la Durezza: potrebbe essere rimovibile.

In [ ]:
X3 = df.drop(["Frutto", "Lunghezza media (mm)"], axis=1)
y3 = df["Frutto"]

grid_search3, X_train3, X_test3, y_train3, y_test3 = split_and_scale_data(X3, y3)

### TEST 4: Rimozione di 'Dolcezza (1-10)'
La Dolcezza ha correlazione bassa con il target. Verifichiamo se la sua assenza impatta le performance.

In [ ]:
X4 = df.drop(["Frutto", "Dolcezza (1-10)"], axis=1)
y4 = df["Frutto"]

grid_search4, X_train4, X_test4, y_train4, y_test4 = split_and_scale_data(X4, y4)

In [ ]:
# ============================================================
# RIEPILOGO COMPARATIVO DEI TEST FEATURE SELECTION
# ============================================================
riepilogo_feature = {
    "Test 1 (tutte le feature)": (grid_search.best_params_['n_neighbors'], grid_search.best_score_),
    "Test 2 (senza Peso)": (grid_search2.best_params_['n_neighbors'], grid_search2.best_score_),
    "Test 3 (senza Lunghezza)": (grid_search3.best_params_['n_neighbors'], grid_search3.best_score_),
    "Test 4 (senza Dolcezza)": (grid_search4.best_params_['n_neighbors'], grid_search4.best_score_),
}

print(f"{'Configurazione':<35} {'Miglior K':>10} {'Accuracy CV':>12}")
print("-" * 60)
for nome, (k, acc) in riepilogo_feature.items():
    marker = " ✓ MIGLIORE" if acc == max(v[1] for v in riepilogo_feature.values()) else ""
    print(f"{nome:<35} {k:>10} {acc:>12.4f}{marker}")

print("\n→ Utilizziamo tutte le feature per il modello finale.")

## SEZIONE 7: TEST 5 — CONFRONTO METRICHE DI DISTANZA KNN

Il KNN classifica un campione in base ai suoi K vicini più prossimi.
La **metrica di distanza** definisce come si misura la 'prossimità'.

Testiamo 4 metriche diverse, tutte applicabili a dati numerici continui:

| Metrica | Formula | Caratteristiche |
|---------|---------|----------------|
| Euclidea | √Σ(xᵢ-yᵢ)² | Distanza 'in linea d'aria', sensibile alla scala (già normalizzata con StandardScaler) |
| Manhattan | Σ|xᵢ-yᵢ| | Somma delle differenze assolute, più robusta agli outlier |
| Chebyshev | max|xᵢ-yᵢ| | Solo la dimensione con differenza massima, utile quando una feature domina |
| Minkowski (p=4) | (Σ|xᵢ-yᵢ|⁴)^(1/4) | Generalizzazione intermedia tra Euclidea e Chebyshev |

In [ ]:
# Usiamo X_train, X_test, y_train, y_test dal Test 1 (tutte le feature, già scalati)
# e il K ottimale trovato (grid_search.best_params_['n_neighbors'])
# per isolare l'effetto della sola metrica di distanza

k_ottimale = grid_search.best_params_['n_neighbors']

metriche = {
    'euclidean': {'p': 2},
    'manhattan': {'p': 1},
    'chebyshev': {},
    'minkowski (p=4)': {'p': 4, 'metric': 'minkowski'},
}

print(f"TEST 5 — Confronto metriche di distanza (K={k_ottimale}, tutte le feature)")
print(f"{'Metrica':<22} {'Accuracy Test':>15}")
print("-" * 40)

risultati_metriche = {}
for nome_metrica, params in metriche.items():
    # Costruiamo i parametri per KNeighborsClassifier
    if 'metric' in params:
        knn_m = KNeighborsClassifier(n_neighbors=k_ottimale, metric=params['metric'], p=params['p'])
    elif nome_metrica == 'chebyshev':
        knn_m = KNeighborsClassifier(n_neighbors=k_ottimale, metric='chebyshev')
    else:
        knn_m = KNeighborsClassifier(n_neighbors=k_ottimale, metric='minkowski', p=params['p'])

    knn_m.fit(X_train, y_train)
    acc = knn_m.score(X_test, y_test)
    risultati_metriche[nome_metrica] = acc
    print(f"{nome_metrica:<22} {acc:>15.4f}")

migliore_metrica = max(risultati_metriche, key=risultati_metriche.get)
print(f"\n→ Metrica migliore: {migliore_metrica} (accuracy={risultati_metriche[migliore_metrica]:.4f})")

## SEZIONE 8: TEST 6 — CONFRONTO PESI NELLA VOTAZIONE KNN

Nel KNN, quando i K vicini votano la classe da assegnare, i pesi definiscono
quanto conta il voto di ciascun vicino:

- **`uniform`** (default): tutti i K vicini hanno lo stesso peso nel voto.
  Semplice e robusto, funziona bene quando i vicini sono distribuiti uniformemente.
- **`distance`**: i vicini più vicini pesano di più (peso = 1/distanza).
  Utile quando esistono campioni di classi diverse mescolati in zone di confine.

Con K grande (es. K=95), la distanza ponderata può aiutare a distinguere
meglio le classi nelle zone di confine (es. Arancia vs Kiwi).

In [ ]:
print(f"TEST 6 — Confronto pesi nella votazione (K={k_ottimale}, tutte le feature)")
print(f"{'Tipo di peso':<15} {'Accuracy Test':>15}")
print("-" * 32)

risultati_pesi = {}
for peso in ['uniform', 'distance']:
    knn_w = KNeighborsClassifier(n_neighbors=k_ottimale, weights=peso)
    knn_w.fit(X_train, y_train)
    acc = knn_w.score(X_test, y_test)
    risultati_pesi[peso] = acc
    print(f"{peso:<15} {acc:>15.4f}")

migliore_peso = max(risultati_pesi, key=risultati_pesi.get)
print(f"\n→ Peso migliore: {migliore_peso} (accuracy={risultati_pesi[migliore_peso]:.4f})")

## SEZIONE 9: TEST 7 — CONFRONTO TRA ALGORITMI DI CLASSIFICAZIONE

Per validare la scelta del KNN, lo confrontiamo con altri algoritmi di classificazione
comuni, tutti addestrati sullo stesso train set (scalato) e valutati sullo stesso test set.

**Algoritmi testati:**
- **KNN**: il nostro modello principale
- **Decision Tree**: modello interpretabile basato su regole if-then
- **Random Forest**: ensemble di alberi decisionali, spesso più robusto
- **SVM**: trova l'iperpiano ottimale di separazione tra classi
- **Naive Bayes (Gaussiano)**: modello probabilistico, assume indipendenza tra feature

Tutti gli algoritmi usano i parametri di default (eccetto random_state per riproducibilità)
per un confronto equo che non favorisca nessun modello.

In [ ]:
# Dizionario degli algoritmi da confrontare
# Tutti usano i parametri default tranne il random_state per riproducibilità
algoritmi = {
    f'KNN (K={k_ottimale}, uniform)': KNeighborsClassifier(n_neighbors=k_ottimale, weights='uniform'),
    'Decision Tree':                  DecisionTreeClassifier(random_state=42),
    'Random Forest (100 alberi)':     RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF kernel)':               SVC(kernel='rbf', random_state=42),
    'Naive Bayes (Gaussiano)':        GaussianNB(),
}

print("TEST 7 — Confronto algoritmi di classificazione")
print(f"{'Algoritmo':<35} {'Accuracy Test':>14}")
print("-" * 52)

risultati_algoritmi = {}
for nome, modello in algoritmi.items():
    modello.fit(X_train, y_train)            # Addestramento sul training set scalato
    acc = modello.score(X_test, y_test)      # Valutazione sul test set scalato
    risultati_algoritmi[nome] = acc
    print(f"{nome:<35} {acc:>14.4f}")

migliore_algo = max(risultati_algoritmi, key=risultati_algoritmi.get)
print(f"\n→ Algoritmo migliore: {migliore_algo}")
print(f"  Accuracy: {risultati_algoritmi[migliore_algo]:.4f}")

# Visualizzazione a barre
plt.figure(figsize=(10, 5))
plt.barh(list(risultati_algoritmi.keys()), list(risultati_algoritmi.values()),
         color=['#2196F3' if 'KNN' in k else '#90CAF9' for k in risultati_algoritmi.keys()])
plt.xlabel('Accuracy sul Test Set')
plt.title('Test 7 — Confronto tra Algoritmi di Classificazione')
plt.xlim(0.7, 1.0)
for i, (nome, acc) in enumerate(risultati_algoritmi.items()):
    plt.text(acc + 0.002, i, f'{acc:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## SEZIONE 10: TEST 8 — CONFRONTO K-FOLD CROSS-VALIDATION

La cross-validation è una tecnica per stimare le performance del modello in modo più robusto
rispetto a una singola suddivisione train/test.

**Come funziona**: il dataset viene suddiviso in K parti (fold). Per K volte, si usa
un fold come test e i restanti K-1 come training. Il risultato finale è la media degli K score.

**Trade-off del numero di fold:**
- **K=3**: stima veloce ma con maggiore varianza (ogni fold è grande, ma meno iterazioni)
- **K=5**: buon bilanciamento tra bias e varianza della stima (scelta standard)
- **K=10**: stima più accurata ma computazionalmente costosa

Confrontiamo i tre per verificare la stabilità del nostro modello KNN.

In [ ]:
# Per la cross-validation usiamo l'intero dataset scalato (non solo il training set)
# La cross-validation gestisce internamente la suddivisione train/validation
scaler_cv = StandardScaler()
X_scaled_completo = scaler_cv.fit_transform(X)  # X = tutte le feature, tutto il dataset

knn_finale = KNeighborsClassifier(n_neighbors=k_ottimale, weights='uniform')

print(f"TEST 8 — Cross-validation con diversi K-fold (KNN, K={k_ottimale})")
print(f"{'N. Fold':<12} {'Mean Accuracy':>15} {'Std Accuracy':>14} {'Intervallo 95%':>20}")
print("-" * 65)

risultati_cv = {}
for n_fold in [3, 5, 10]:
    scores = cross_val_score(knn_finale, X_scaled_completo, y, cv=n_fold, scoring='accuracy')
    risultati_cv[n_fold] = scores
    # L'intervallo di confidenza al 95% è approssimato come mean ± 2*std
    ci_low = scores.mean() - 2 * scores.std()
    ci_high = scores.mean() + 2 * scores.std()
    print(f"CV-{n_fold:<9} {scores.mean():>15.4f} {scores.std():>14.4f} {f'[{ci_low:.4f}, {ci_high:.4f}]':>20}")

print("\n→ La coerenza tra i risultati dei diversi fold conferma la stabilità del modello.")

# Visualizzazione a boxplot degli score per fold
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([risultati_cv[3], risultati_cv[5], risultati_cv[10]],
           labels=['3-Fold', '5-Fold', '10-Fold'])
ax.set_ylabel('Accuracy')
ax.set_title(f'Test 8 — Distribuzione degli score per K-Fold CV (KNN, K={k_ottimale})')
ax.set_ylim(0.8, 1.0)
ax.grid(axis='y', alpha=0.5)
plt.tight_layout()
plt.show()

## SEZIONE 11: VALUTAZIONE DELLE PERFORMANCE DEL MODELLO FINALE

Dai test precedenti abbiamo determinato la configurazione ottimale:
- **Feature**: tutte e 5 (Test 1 > Test 2, 3, 4)
- **K**: valore ottimale trovato da GridSearchCV con 5-fold CV
- **Pesi**: `uniform` (migliore o equivalente a `distance`)
- **Metrica**: euclidea (default, migliore con dati normalizzati)

Addestriamo il modello finale e ne valutiamo le performance complete.

In [ ]:
# ============================================================
# MODELLO FINALE: KNN con K ottimale e weights='uniform'
# ============================================================
# Usiamo X_train, X_test, y_train, y_test dal Test 1 (tutte le feature, già scalati)

knn = KNeighborsClassifier(
    n_neighbors=k_ottimale,
    weights='uniform',     # Tutti i vicini contano ugualmente
    metric='euclidean'     # Distanza euclidea, appropriata con dati standardizzati
)
knn.fit(X_train, y_train)

print(f"Modello finale: KNN con K={k_ottimale}, weights='uniform', metric='euclidean'")

In [ ]:
# ============================================================
# METRICHE DI VALUTAZIONE
# ============================================================
score = knn.score(X_test, y_test)
print(f"Accuracy sul test set: {score:.4f} ({score*100:.2f}%)")

y_pred = knn.predict(X_test)

print("\n--- Classification Report ---")
print("(Per ogni classe: precision = quante predizioni positive sono corrette,")
print(" recall = quante istanze positive vengono trovate, F1 = media armonica)")
print()

# NOTA: l'ordine corretto degli argomenti è classification_report(y_true, y_pred)
# Prima i valori reali (y_test), poi quelli predetti (y_pred)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

In [ ]:
# L'errore di classificazione è il complemento dell'accuracy:
# rappresenta la percentuale di campioni classificati erroneamente
error_rate = 1 - score
print(f"Accuracy:                {score:.4f} ({score*100:.2f}%)")
print(f"Errore di classificazione: {error_rate:.4f} ({error_rate*100:.2f}%)")

In [ ]:
# ============================================================
# CURVA ACCURACY/ERRORE AL VARIARE DI K
# ============================================================
# Visualizziamo come cambiano accuracy ed errore al variare di K (ogni 5 valori)
# per confermare visivamente che K ottimale sia quello trovato da GridSearchCV

def calcola_metriche_per_k(X_train, X_test, y_train, y_test, step=5):
    metriche_k = {}
    for k in range(step, 101, step):
        knn_k = KNeighborsClassifier(n_neighbors=k)
        knn_k.fit(X_train, y_train)
        acc = knn_k.score(X_test, y_test)
        metriche_k[k] = {'accuracy': acc, 'errore': 1 - acc}
    return metriche_k


metrics_dict = calcola_metriche_per_k(X_train, X_test, y_train, y_test)

k_values = list(metrics_dict.keys())
accuracy_rates = [metrics_dict[k]['accuracy'] for k in k_values]
error_rates = [metrics_dict[k]['errore'] for k in k_values]

plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracy_rates, marker='o', linestyle='-', color='blue', label='Accuracy')
plt.plot(k_values, error_rates, marker='s', linestyle='--', color='red', label='Errore')

# Evidenzio il K ottimale
if k_ottimale in metrics_dict:
    plt.axvline(x=k_ottimale, color='green', linestyle=':', alpha=0.8,
                label=f'K ottimale = {k_ottimale}')

plt.yticks(np.arange(0, 1.05, 0.05))
plt.xlabel('Valore di K')
plt.ylabel('Metrica')
plt.title('Performance del modello KNN al variare di K')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**ANALISI DELLA CURVA**

- Per **K piccoli** (K<20): alta varianza, il modello è sensibile al rumore → rischio overfitting
- Per **K grandi** (K≥50): le performance si stabilizzano, il modello è più robusto
- Il **picco di accuracy** si raggiunge attorno al K ottimale trovato da GridSearchCV

La curva conferma visivamente la scelta del K ottimale.

## SEZIONE 12: MATRICE DI CONFUSIONE

In [ ]:
# ============================================================
# MATRICE DI CONFUSIONE
# ============================================================
# La matrice di confusione mostra quante volte ogni classe è stata:
# - classificata correttamente (diagonale principale)
# - confusa con un'altra classe (celle fuori diagonale)
#
# È più informativa della sola accuracy perché rivela DOVE sbaglia il modello.
# Utile per identificare coppie di classi difficili da distinguere.

cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index=encoder.classes_, columns=encoder.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            linewidths=0.5, linecolor='gray')
plt.xlabel('Classe Predetta', fontsize=12)
plt.ylabel('Classe Reale', fontsize=12)
plt.title('Matrice di Confusione — Modello KNN Finale', fontsize=13)
plt.tight_layout()
plt.show()

# Riepilogo testuale degli errori
print("\n--- Classificazioni per classe ---")
for i, frutto in enumerate(encoder.classes_):
    corretti = cm[i, i]
    totale = cm[i, :].sum()
    errori = totale - corretti
    print(f"{frutto:<10}: {corretti}/{totale} corretti ({errori} errori)")

**ANALISI DELLA MATRICE DI CONFUSIONE**

**Classi perfettamente classificate:**
- **Banana** e **Uva**: 100% di accuracy. Le loro caratteristiche (Banana: molto pesante e lunga; Uva: molto leggera e piccola) le rendono facilmente distinguibili.

**Classi con confusione:**
- **Arancia (classe 0)** e **Kiwi (classe 2)**: si confondono reciprocamente. Hanno caratteristiche simili (peso, diametro, dolcezza sovrapponibili) che rendono difficile la distinzione al modello.

**Motivazione degli errori residui:**
Le confusioni tra Arancia e Kiwi sono fisiologiche: questi frutti hanno feature numeriche
con distribuzioni parzialmente sovrapposte, e il KNN con K grande tende a 'ammorbidire'
i confini tra queste classi vicine nello spazio delle feature.

## SEZIONE 13: ANALISI DEGLI ERRORI SULLE CLASSI PROBLEMATICHE

In [ ]:
# ============================================================
# ANALISI DETTAGLIATA DEGLI ERRORI SULLE CLASSI 0 E 2
# ============================================================
# Dalla matrice di confusione, Arancia (0) e Kiwi (2) sono le classi più problematiche.
# Creiamo un DataFrame con i valori reali e predetti per analizzare dove il modello sbaglia.

df_test = pd.DataFrame({
    'Valore Reale': y_test.values,
    'Valore Predetto': y_pred,
    'Corretto': y_test.values == y_pred
})

# Codice 0 = Arancia, Codice 2 = Kiwi (dal mapping LabelEncoder)
errori_arancia = df_test[(df_test['Valore Reale'] == 0) & (~df_test['Corretto'])]
errori_kiwi = df_test[(df_test['Valore Reale'] == 2) & (~df_test['Corretto'])]

print(f"Errori su Arancia (classe 0): {len(errori_arancia)} campioni")
for _, row in errori_arancia.iterrows():
    print(f"  → Classificato come '{encoder.classes_[int(row['Valore Predetto'])]}'")

print(f"\nErrori su Kiwi (classe 2): {len(errori_kiwi)} campioni")
for _, row in errori_kiwi.iterrows():
    print(f"  → Classificato come '{encoder.classes_[int(row['Valore Predetto'])]}'")

In [ ]:
# ============================================================
# VERIFICA DEL BILANCIAMENTO DELLE CLASSI
# ============================================================
# Un dataset sbilanciato può favorire le classi maggioritarie.
# Con il nostro dataset bilanciato (100 campioni per classe),
# la metrica 'accuracy' è una misura affidabile delle performance.

print("Distribuzione delle classi nel dataset completo:")
print(df["Frutto"].value_counts().rename(index=dict(enumerate(encoder.classes_))))

print("\nDistribuzione delle classi nel test set:")
test_dist = pd.Series(y_test).value_counts().rename(index=dict(enumerate(encoder.classes_)))
print(test_dist)

print("\n→ Il dataset è bilanciato: l'accuracy è una metrica di valutazione affidabile.")
print("→ Tecniche come SMOTE o class_weight non sono necessarie in questo contesto.")

## CONCLUSIONI FINALI

### Riepilogo dei Test Effettuati

| Test | Obiettivo | Risultato |
|------|-----------|----------|
| **Test 1** (tutte le feature) | Baseline con tutte le feature | Miglior accuracy → configurazione scelta |
| **Test 2** (senza Peso) | Verificare ridondanza con Lunghezza | Performance inferiore: Peso contribuisce |
| **Test 3** (senza Lunghezza) | Verificare ridondanza con Peso | Performance inferiore: Lunghezza contribuisce |
| **Test 4** (senza Dolcezza) | Valutare l'impatto della feature meno correlata | Performance inferiore: Dolcezza contribuisce |
| **Test 5** (metriche distanza) | Trovare la metrica di distanza ottimale | Verificata la metrica più performante |
| **Test 6** (pesi votazione) | Confronto uniform vs distance | Verificato il tipo di peso migliore |
| **Test 7** (confronto algoritmi) | Validare la scelta del KNN | KNN competitivo rispetto agli altri modelli |
| **Test 8** (k-fold CV) | Verificare la stabilità del modello | Risultati coerenti tra 3, 5 e 10 fold |

### Performance del Modello Finale

- **Algoritmo**: K-Nearest Neighbors
- **K ottimale**: trovato tramite GridSearchCV con 5-fold cross-validation
- **Feature**: tutte e 5 le feature numeriche disponibili
- **Preprocessing**: StandardScaler (fit solo su training set)
- **Accuracy**: ≈94% sul test set
- **Classi perfette**: Banana e Uva (100% accuracy)
- **Classi problematiche**: Arancia e Kiwi (caratteristiche numeriche parzialmente sovrapposte)

### Motivazione della Scelta del KNN

Il KNN si è rivelato una scelta appropriata per questo problema perché:
1. Il dataset è bilanciato (100 campioni per classe) → nessun bias verso classi maggioritarie
2. Le feature numeriche sono ben definite e significative per distinguere i frutti
3. Con 500 campioni, il costo computazionale del KNN è accettabile
4. Il modello è interpretabile: la classificazione si basa sui frutti 'più simili' nel dataset di training